# 10 — Integrated Risk Engine (ICRRI)
**Objective:** Demonstrate and validate the ICRRI computation logic.

In [ ]:
import sys; sys.path.insert(0,'../backend')
import os; os.environ['DATABASE_URL']='sqlite:///test.db'; os.environ['SECRET_KEY']='test-key-32chars-x'
from app.clinical.engines import compute_icrri, evaluate_rules

## ICRRI Computation Examples

In [ ]:
test_cases = [
    (0.0, 0.0, 0.0, 'All zero → LOW'),
    (1.0, 1.0, 1.0, 'All max → CRITICAL'),
    (0.7, 0.5, 0.6, 'High CKD → HIGH/CRITICAL'),
    (0.2, 0.2, 0.2, 'Low risk → LOW'),
    (0.4, 0.4, 0.4, 'Moderate → MODERATE'),
]
print(f"{'CKD':>6} {'DM':>6} {'HTN':>6}  {'ICRRI':>7}  {'Category'}")
print('-'*50)
for ckd, dm, htn, note in test_cases:
    icrri, cat, _ = compute_icrri(ckd, dm, htn)
    print(f"{ckd:6.1f} {dm:6.1f} {htn:6.1f}  {icrri:7.2f}  {cat:10s} ← {note}")

## ICRRI Sensitivity Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

probs = np.linspace(0, 1, 20)
results = [compute_icrri(p, 0.5, 0.5)[0] for p in probs]
plt.figure(figsize=(8,4))
plt.plot(probs*100, results, 'b-o', markersize=4)
plt.xlabel('CKD Probability (%)')
plt.ylabel('ICRRI')
plt.title('ICRRI vs CKD Probability (DM=50%, HTN=50%)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Clinical Rule Engine Test

In [ ]:
test_measurements = {'potassium': 6.2, 'systolic_bp': 185,
                       'creatinine': 2.5, 'hba1c': 9.0}
alerts = evaluate_rules(test_measurements)
print(f"Triggered {len(alerts)} alerts:")
for a in alerts:
    print(f"  [{a['severity']:12s}] {a['rule_id']}: {a['title']}")

## Disclaimer
The ICRRI is a **project-specific composite screening index**. It is NOT a clinically validated medical score. Weights and thresholds are project configuration, not evidence-based clinical guidelines.